# Klyro-V1000 Training Notebook\n\nThis notebook runs the full training pipeline for Klyro-V1000 on Kaggle.\nIt assumes you have set up:\n- A Hugging Face repository (e.g., `username/Klyro-V1000`)\n- A Hugging Face access token stored in Kaggle Secrets as `HF_TOKEN`\n- The repository ID configured in `config/train_config.yaml`\n

In [ ]:
# 1. Check GPU\nimport subprocess\nimport sys\ndef run_cmd(cmd):\n    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)\n    return result.stdout.strip(), result.stderr.strip(), result.returncode\n\nout, err, code = run_cmd('nvidia-smi')\nprint('GPU Info:')\nprint(out if out else 'No GPU found')\n\n# 2. Install dependencies (if not already)\n!pip install -q -r /kaggle/working/Klyro-V1000/requirements.txt

In [ ]:
# 3. Load configuration\nimport yaml\nwith open('/kaggle/working/Klyro-V1000/config/train_config.yaml', 'r') as f:\n    config = yaml.safe_load(f)\nprint('Configuration loaded')\nprint(f"HF Repo ID: {config['huggingface']['repo_id']}")\nprint(f"Max steps: {config['training']['max_steps']}")

In [ ]:
# 4. Get HF token from Kaggle Secrets\nimport os\nfrom kaggle_secrets import UserSecretsClient\nuser_secrets = UserSecretsClient()\nhf_token = user_secrets.get_secret("HF_TOKEN")\nos.environ["HF_TOKEN"] = hf_token\nprint('HF token loaded (length:', len(hf_token), ')')

In [ ]:
# 5. Ensure HF repo exists\nfrom src.hf_utils import ensure_repo_exists\nensure_repo_exists(config['huggingface']['repo_id'], token=hf_token)\nprint('Repository ready')

In [ ]:
# 6. Determine fresh vs resume\nstart_from_scratch = config.get('checkpoint', {}).get('start_from_scratch', True)\nresume = config.get('checkpoint', {}).get('resume_from_checkpoint', False)\nprint(f'Start from scratch: {start_from_scratch}')\nprint(f'Resume from checkpoint: {resume}')

In [ ]:
# 7. Load datasets (streaming) - placeholders\nfrom datasets import load_dataset, interleave_datasets\n# English\neng = load_dataset(config['data']['english_dataset'], name=config['data']['english_config'], split='train', streaming=True)\n# Russian\nrus = load_dataset(config['data']['russian_dataset'], name=config['data']['russian_config'], split='train', streaming=True)\n# Mix\neng_prob = 1.0 - config['data']['russian_ratio']\nrus_prob = config['data']['russian_ratio']\nmixed = interleave_datasets([eng, rus], probabilities=[eng_prob, rus_prob], seed=config['training']['seed'])\nprint('Datasets prepared')

In [ ]:
# 8. Tokenizer training or loading\nimport os\nfrom src.tokenizer import KlyroTokenizer\ntokenizer = KlyroTokenizer(vocab_size=config['model']['vocab_size'])\ntokenizer_dir = '/kaggle/working/Klyro-V1000/tokenizer'\nif os.path.isdir(tokenizer_dir) and os.path.exists(os.path.join(tokenizer_dir, 'tokenizer.json')):\n    tokenizer.load(tokenizer_dir)\n    print('Tokenizer loaded from disk')\nelse:\n    # Train on a sample of the mixed dataset\n    def sample_iterator(dataset, n=10000):\n        count = 0\n        for example in dataset:\n            yield example['text']\n            count += 1\n            if count >= n:\n                break\n    print('Training tokenizer on sample...')\n    tokenizer.train(sample_iterator(mixed, n=20000))\n    os.makedirs(tokenizer_dir, exist_ok=True)\n    tokenizer.save(tokenizer_dir)\n    print('Tokenizer trained and saved')

In [ ]:
# 9. Model creation\nimport torch\nfrom src.model import KlyroConfig, KlyroForCausalLM\ncfg = KlyroConfig(\n    vocab_size=config['model']['vocab_size'],\n    hidden_size=config['model']['hidden_size'],\n    num_hidden_layers=config['model']['num_hidden_layers'],\n    num_attention_heads=config['model']['num_attention_heads'],\n    intermediate_size=config['model']['intermediate_size'],\n    max_position_embeddings=config['model']['max_position_embeddings'],\n    hidden_act=config['model']['hidden_act'],\n    layer_norm_eps=config['model']['layer_norm_eps'],\n    initializer_range=config['model']['initializer_range'],\n    pad_token_id=tokenizer.token_to_id('[PAD]'),\n)\nmodel = KlyroForCausalLM(cfg)\nprint(f"Model parameters: {model.num_parameters():,}")\nmodel.to('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 10. Training loop (simplified call to src.train.train_loop)\nimport sys\nsys.path.append('/kaggle/working/Klyro-V1000/src')\nfrom src.train import train_loop\nfrom src.dataset import get_language_datasets, mix_datasets, prepare_dataloader\nimport torch.optim as optim\nfrom transformers import get_linear_schedule_with_warmup\n\n# Prepare dataloaders (same as in notebook for brevity)\ntrain_datasets = get_language_datasets(config, split='train')\neng_prob = 1.0 - config['data']['russian_ratio']\nrus_prob = config['data']['russian_ratio']\nmixed_train = mix_datasets(train_datasets, probabilities=[eng_prob, rus_prob], seed=config['training']['seed'])\ntrain_loader = prepare_dataloader(mixed_train, tokenizer, batch_size=config['training']['batch_size'], max_length=config['model']['max_position_embeddings'])\n\neval_datasets = get_language_datasets(config, split='validation')\nmixed_eval = mix_datasets(eval_datasets, probabilities=[eng_prob, rus_prob], seed=config['training']['seed'])\neval_loader = prepare_dataloader(mixed_eval, tokenizer, batch_size=config['training']['batch_size'], max_length=config['model']['max_position_embeddings'], shuffle=False) if eval_datasets else None\n\noptimizer = optim.AdamW(model.parameters(), lr=config['training']['learning_rate'], weight_decay=config['training']['weight_decay'])\nscheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=config['training']['warmup_steps'], num_training_steps=config['training']['max_steps'])\n\n# Call training loop\ntrain_loop(model, tokenizer, optimizer, scheduler, train_loader, eval_loader, config, device='cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 11. After training: generate sample texts\nfrom src.inference import load_model_and_tokenizer, generate\nmodel, tokenizer = load_model_and_tokenizer(repo_id=config['huggingface']['repo_id'], token=hf_token, device='cuda' if torch.cuda.is_available() else 'cpu')\nprint('Russian generation:')\nprint(generate(model, tokenizer, "Привет, как дела?", max_length=50, temperature=0.8, top_k=50, device='cuda' if torch.cuda.is_available() else 'cpu'))\nprint('\nEnglish generation:')\nprint(generate(model, tokenizer, "Hello, how are you?", max_length=50, temperature=0.8, top_k=50, device='cuda' if torch.cuda.is_available() else 'cpu'))